# Person Attributes Training (Gender + Age)
## Rock Cluster Camera Brain Project

Multi-task ResNet18 for person attribute prediction:
- **Gender**: Male / Female
- **Age Range**: Child, Teen, Adult, Senior

**GPU Required:** Runtime → Change runtime type → GPU (T4)

In [ ]:
# Check GPU
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# Install dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install albumentations pandas tqdm pillow matplotlib
!pip install onnx

In [ ]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pandas as pd
import numpy as np
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
import onnx
import os

## Dataset Setup

Expected format:
```
/content/data/
  ├── train.csv  # columns: image_path, gender, age_range
  ├── val.csv
  └── images/
      ├── img_001.jpg
      └── ...
```

gender: 0=Male, 1=Female
age_range: 0=Child, 1=Teen, 2=Adult, 3=Senior

In [ ]:
# Create demo dataset (for testing pipeline)
# Replace this with actual PETA/PA100K dataset loading

!mkdir -p /content/data/images

# Create synthetic CSV files
import random

def create_demo_dataset(n_train=500, n_val=100):
    # Download some sample person images from open datasets
    # For demo, we create placeholder entries
    
    train_data = []
    val_data = []
    
    # This is a placeholder - actual dataset loading would go here
    print("DEMO MODE: Creating synthetic dataset for pipeline testing")
    print("For real training, mount Google Drive with PETA/PA100K dataset")
    
    # Create dummy entries (will be replaced with actual data loading)
    for i in range(n_train):
        train_data.append({
            'image_path': f'/content/data/images/demo_{i}.jpg',
            'gender': random.randint(0, 1),
            'age_range': random.randint(0, 3)
        })
    
    for i in range(n_val):
        val_data.append({
            'image_path': f'/content/data/images/demo_{i}.jpg',
            'gender': random.randint(0, 1),
            'age_range': random.randint(0, 3)
        })
    
    pd.DataFrame(train_data).to_csv('/content/data/train.csv', index=False)
    pd.DataFrame(val_data).to_csv('/content/data/val.csv', index=False)
    
    print(f"Created demo CSV with {n_train} train / {n_val} val samples")
    return train_data, val_data

create_demo_dataset()

In [ ]:
# Dataset class for person attributes
class PersonAttributesDataset(Dataset):
    def __init__(self, csv_path, transform=None, demo_mode=False):
        self.transform = transform
        self.demo_mode = demo_mode
        
        if os.path.exists(csv_path):
            self.df = pd.read_csv(csv_path)
        else:
            self.df = pd.DataFrame(columns=['image_path', 'gender', 'age_range'])
        
        print(f"Loaded {len(self.df)} samples from {csv_path}")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # For demo mode, create random tensor
        if self.demo_mode or not os.path.exists(row['image_path']):
            img = np.random.randint(0, 255, (256, 128, 3), dtype=np.uint8)
        else:
            img = np.array(Image.open(row['image_path']).convert('RGB'))
            img = A.Resize(256, 128)(image=img)['image']
        
        if self.transform:
            img = self.transform(image=img)['image']
        
        gender = torch.tensor(row['gender'], dtype=torch.float32)
        age = torch.tensor(row['age_range'], dtype=torch.long)
        
        return img, gender, age

# Transforms
train_transform = A.Compose([
    A.Resize(256, 128),
    A.RandomCrop(224, 112),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(224, 112),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

In [ ]:
# Multi-task Person Attribute Network
class PersonAttributeNet(nn.Module):
    def __init__(self, num_age_classes=4):
        super().__init__()
        
        # ResNet18 backbone
        resnet = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.feature_dim = 512
        
        # Gender head (binary)
        self.gender_head = nn.Sequential(
            nn.Linear(self.feature_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )
        
        # Age head (4 classes)
        self.age_head = nn.Sequential(
            nn.Linear(self.feature_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_age_classes)
        )
    
    def forward(self, x):
        features = self.features(x).view(-1, self.feature_dim)
        gender_logits = self.gender_head(features)
        age_logits = self.age_head(features)
        return gender_logits, age_logits

NUM_AGE_CLASSES = 4
model = PersonAttributeNet(NUM_AGE_CLASSES)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Model created: ResNet18-based multi-task network")
print(f"Gender head: Binary (Male/Female)")
print(f"Age head: {NUM_AGE_CLASSES}-class (Child/Teen/Adult/Senior)")
print(f"Device: {device}")

In [ ]:
# Create data loaders
BATCH_SIZE = 32

train_dataset = PersonAttributesDataset('/content/data/train.csv', transform=train_transform, demo_mode=True)
val_dataset = PersonAttributesDataset('/content/data/val.csv', transform=val_transform, demo_mode=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
# Loss functions and optimizer
gender_criterion = nn.BCEWithLogitsLoss()
age_criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

def train_epoch(model, loader, optimizer, device):
    model.train()
    gender_loss_sum, age_loss_sum = 0, 0
    gender_correct, age_correct, total = 0, 0, 0
    
    for images, genders, ages in tqdm(loader, desc='Training'):
        images = images.to(device)
        genders = genders.to(device).unsqueeze(1)
        ages = ages.to(device)
        
        optimizer.zero_grad()
        gender_out, age_out = model(images)
        
        gender_loss = gender_criterion(gender_out, genders)
        age_loss = age_criterion(age_out, ages)
        loss = gender_loss + age_loss
        
        loss.backward()
        optimizer.step()
        
        gender_loss_sum += gender_loss.item()
        age_loss_sum += age_loss.item()
        
        gender_pred = (torch.sigmoid(gender_out) > 0.5).float()
        gender_correct += (gender_pred == genders).sum().item()
        
        age_pred = age_out.max(1)[1]
        age_correct += (age_pred == ages).sum().item()
        
        total += images.size(0)
    
    return (
        gender_loss_sum / len(loader),
        age_loss_sum / len(loader),
        gender_correct / total,
        age_correct / total
    )

def validate(model, loader, device):
    model.eval()
    gender_loss_sum, age_loss_sum = 0, 0
    gender_correct, age_correct, total = 0, 0, 0
    
    with torch.no_grad():
        for images, genders, ages in tqdm(loader, desc='Validating'):
            images = images.to(device)
            genders = genders.to(device).unsqueeze(1)
            ages = ages.to(device)
            
            gender_out, age_out = model(images)
            
            gender_loss = gender_criterion(gender_out, genders)
            age_loss = age_criterion(age_out, ages)
            
            gender_loss_sum += gender_loss.item()
            age_loss_sum += age_loss.item()
            
            gender_pred = (torch.sigmoid(gender_out) > 0.5).float()
            gender_correct += (gender_pred == genders).sum().item()
            
            age_pred = age_out.max(1)[1]
            age_correct += (age_pred == ages).sum().item()
            
            total += images.size(0)
    
    return (
        gender_loss_sum / len(loader),
        age_loss_sum / len(loader),
        gender_correct / total,
        age_correct / total
    )

In [ ]:
# Training loop
NUM_EPOCHS = 30
best_gender_acc = 0
history = {'gender_loss': [], 'age_loss': [], 'gender_acc': [], 'age_acc': []}

print(f"Training for {NUM_EPOCHS} epochs...")

for epoch in range(NUM_EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
    
    g_loss, a_loss, g_acc, a_acc = train_epoch(model, train_loader, optimizer, device)
    val_g_loss, val_a_loss, val_g_acc, val_a_acc = validate(model, val_loader, device)
    
    scheduler.step()
    
    history['gender_loss'].append(g_loss)
    history['age_loss'].append(a_loss)
    history['gender_acc'].append(g_acc)
    history['age_acc'].append(a_acc)
    
    print(f"Train: G_Loss={g_loss:.4f} A_Loss={a_loss:.4f} | G_Acc={g_acc:.4f} A_Acc={a_acc:.4f}")
    print(f"Val:   G_Loss={val_g_loss:.4f} A_Loss={val_a_loss:.4f} | G_Acc={val_g_acc:.4f} A_Acc={val_a_acc:.4f}")
    
    if val_g_acc > best_gender_acc:
        best_gender_acc = val_g_acc
        torch.save({
            'model_state_dict': model.state_dict(),
            'gender_acc': val_g_acc,
            'age_acc': val_a_acc,
            'num_age_classes': NUM_AGE_CLASSES
        }, '/content/person_attr_best.pth')
        print(f"✓ Best model saved! Gender Acc: {best_gender_acc:.4f}")

print(f"\n✓ Training complete! Best gender accuracy: {best_gender_acc:.4f}")

In [ ]:
# Plot training curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['gender_loss'], label='Gender Loss')
plt.plot(history['age_loss'], label='Age Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss Curves')

plt.subplot(1, 2, 2)
plt.plot(history['gender_acc'], label='Gender Acc')
plt.plot(history['age_acc'], label='Age Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy Curves')

plt.tight_layout()
plt.savefig('/content/person_attr_curves.png', dpi=150)
plt.show()

In [ ]:
# Export to ONNX
print("Exporting to ONNX...")

model.eval()
dummy_input = torch.randn(1, 3, 224, 112).to(device)

torch.onnx.export(
    model,
    dummy_input,
    '/content/person_attr.onnx',
    input_names=['input'],
    output_names=['gender', 'age'],
    dynamic_axes={'input': {0: 'batch_size'}, 'gender': {0: 'batch_size'}, 'age': {0: 'batch_size'}},
    opset_version=11
)

# Verify
onnx_model = onnx.load('/content/person_attr.onnx')
onnx.checker.check_model(onnx_model)
print("✓ ONNX model exported and verified!")

# Save config
config = {
    'model_type': 'resnet18_multi_task',
    'input_size': [224, 112],
    'gender_classes': ['male', 'female'],
    'age_classes': ['child', 'teen', 'adult', 'senior'],
    'num_age_classes': NUM_AGE_CLASSES
}

with open('/content/person_attr_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("✓ Config saved!")

In [ ]:
# Download
from google.colab import files

files.download('/content/person_attr_best.pth')
files.download('/content/person_attr.onnx')
files.download('/content/person_attr_config.json')
files.download('/content/person_attr_curves.png')

print("\n✓ Download complete!")
print("Next: Convert ONNX to RKNN using rknn_conversion.py")